In [1]:
import numpy as np
import numba
import time
import concurrent.futures
from numpy.random import Generator, PCG64DXSM, SeedSequence
from math import floor
from sys import float_info
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import ipynbname

from FNs_genBachStravinsky_v0055k05_BZforget3_assort_f7RothErev_rep_execNULLsig import genBS_f7_RothErevExec_full_play_typed, population_array, mutual_info, average_mutual_info, getalphasignaling

np.set_printoptions(suppress=True)
np.set_printoptions(threshold=np.inf)


In [2]:
runpass = ipynbname.name()
thrds = 18

runs = 100
runlength = 4*10**4
record_interval = runlength
signal_snapshots = 1 # need this to be 1 for correct sweep measures of the presence of the desired topology

population_size = 5*10**2
numtypes = 3
mutation_rate = np.array([0.01, 0.1])
# percent_agents_per_type = [.5, .5]
sigdimensions = 2
numsignals_perdim = np.array([2, 2])
numsignals = np.prod(numsignals_perdim)
# print(numsignals)
sigmultipliers = np.cumprod(numsignals_perdim)
sigmultipliers = np.roll(sigmultipliers, 1)
sigmultipliers[0] = 1
# print(sigmultipliers)



base_connection_weights = np.array([1, 2, 4, 8, 16, 32]) #this should be at least one longer than the number of sig dimensions
# reinforcement = 10
# punishment = -2
numactions = 3
# coordination_preferences = np.array([[1, 1], [1, 1]])
genBSpunish = np.ones((2))*0
signal_cost = -0.000125 # this should be a negative value if not 0
BZforget_multiplier = 0.998
inertia = 1
homophily_factor = 0
epsilon = float_info.epsilon #add this value to random uniform distribution to get (0, 1] instead of [0, 1)

# repmultipliers is like sigmultipliers, but maps entire strategy profile for both senders and receivers to a unique value
# modifying this to add in executives after receivers
dimactions= list(numsignals_perdim)
for dadex in range(0, numsignals):
    dimactions.append(numactions)
# for dadex2 in range(0, sigdimensions): # this is added in for executives # ***** removed for 54b
#     dimactions.append(2) # appending 2 because executive is always 0 for ignore dimension or 1 for attend dimension
dimactions = np.array(dimactions)
numprofiles = np.prod(dimactions)
print(numprofiles)
repmultipliers = np.cumprod(dimactions)
repmultipliers = np.roll(repmultipliers, 1)
repmultipliers[0] = 1
print(repmultipliers)



324
[  1   2   4  12  36 108]


In [3]:
# ________________________________________________________________________________________________
t0sigs = [[0, 1], [1, 0]]
t1sigs = [1, 1]
print(t0sigs)
print(t1sigs)
print(' ')
t0sigs_index = []
t1sigs_index = []
for intdex3 in range(0, len(t0sigs)):
    t0s = 0
    t1s = 0
    for intdex4 in range(0, sigdimensions):
        t0s += t0sigs[intdex3][intdex4]*sigmultipliers[intdex4]
        t1s += t1sigs[intdex4]*sigmultipliers[intdex4]
    t0sigs_index.append(t0s)
    t1sigs_index.append(t1s)
print(t0sigs_index)
print(t1sigs_index)

[[0, 1], [1, 0]]
[1, 1]
 
[np.int64(2), np.int64(1)]
[np.int64(3), np.int64(3)]


In [4]:

payoff_alphas = [0.4, 0.6, 0.8]
cents_offset = [-0.03, 0, 0.03]

concise0array = np.zeros((3, 3))
concise1array = np.zeros((3, 3))
concise2array = np.zeros((3, 3))

for dex0 in range(0, 3):
    for dex1 in range(0, 3):
        start = time.perf_counter() # starting a timer to see how long it takes simulation to run

        pa = payoff_alphas[dex0]
        co = cents_offset[dex1]

        percent_agents_per_type = [.33+co, .33-co, .34]

        coordination_preferences = (np.array([[1.5, 0, .85], 
                                              [1.25, 1.5+pa, .85], 
                                              [0, 0, 1]]))/200


        sq1 = SeedSequence()
        randomentropy = sq1.entropy
        # randomentropy = 198033477696915839319494177136675262199
        print(randomentropy)
        sg = SeedSequence(randomentropy)
        rgs = numba.typed.List([Generator(PCG64DXSM(s)) for s in sg.spawn(runs)])


        sigurns = []
        for idx0000 in range(0, sigdimensions):
            sigurns.append(np.ones([population_size, numsignals_perdim[idx0000]])*inertia)

        recurns = np.ones([population_size, numsignals, numactions])*inertia
        popt = population_array(population_size, percent_agents_per_type)

        uniquepopt, countspopt = np.unique(popt, return_counts=True)
        print(np.asarray((uniquepopt, countspopt)).T)


        # initiallizing some arrays to store results in
        final_simple_stat_001 = np.zeros((runs, numtypes, numtypes, numactions), dtype=np.int64)
        final_simple_stat_002 = np.zeros((runs, numtypes, numtypes, numactions), dtype=np.int64)
        final_socialsig_aggregate = np.zeros((runs, signal_snapshots, numtypes, numsignals), dtype=np.int64)
        final_action_aggregate = np.zeros((runs, numtypes, numsignals, numactions), dtype=np.int64)
        final_sigurns = []
        final_recurns = []
        final_execurns = np.zeros((runs, population_size, sigdimensions), dtype=np.int64)
        final_typed_time = np.zeros((runs, numtypes, numsignals, numactions, runlength//record_interval), dtype=np.float64)
        final_signal_time = np.zeros((runs, numsignals, numsignals, numactions, runlength//record_interval), dtype=np.float64)
        final_typed_time_norm = np.zeros((runs, numtypes, numsignals, numactions, runlength//record_interval), dtype=np.float64)
        final_signal_time_norm = np.zeros((runs, numsignals, numsignals, numactions, runlength//record_interval), dtype=np.float64)
        for final_idx in range(0, runs):
            final_sigurns.append(sigurns)
            final_recurns.append(recurns)

        inputs = []

        with concurrent.futures.ProcessPoolExecutor(max_workers=thrds) as executor:
            future_to_genBS = {executor.submit(genBS_f7_RothErevExec_full_play_typed, BZforget_multiplier, mutation_rate, signal_cost, repmultipliers, numprofiles, signal_snapshots, record_interval, genBSpunish, numtypes, numsignals, runlength, numactions, coordination_preferences, popt, sigurns, recurns, population_size, sigdimensions, base_connection_weights, sigmultipliers, homophily_factor, rgs[r], epsilon, r): inputs for r in range(runs)}
            for future in concurrent.futures.as_completed(future_to_genBS):
                inputs = future_to_genBS[future]
                try:
                    data_simple_stat_002, data_simple_stat_001, data_socialsig_aggregate, data_action_aggregate, data_sigurns, data_recurns, data_execurns, data_typed_time, data_signal_time, data_typed_time_norm, data_signal_time_norm, data_runid = future.result()
                except Exception as exc:
                    print(f'generated an exception: y? and {exc}')
                else:
                    final_simple_stat_001[data_runid] = data_simple_stat_001
                    final_simple_stat_002[data_runid] = data_simple_stat_002
                    final_socialsig_aggregate[data_runid] = data_socialsig_aggregate
                    final_action_aggregate[data_runid] = data_action_aggregate
                    final_sigurns[data_runid] = data_sigurns
                    final_recurns[data_runid] = data_recurns
                    final_execurns[data_runid] = data_execurns
                    final_typed_time[data_runid] = data_typed_time
                    final_signal_time[data_runid] = data_signal_time
                    final_typed_time_norm[data_runid] = data_typed_time_norm
                    final_signal_time_norm[data_runid] = data_signal_time_norm
                    if data_runid%10 == 0:
                        print(f'finished run #{data_runid}')

#         for testruns in range(0, runs):
#             (data_simple_stat_002, data_simple_stat_001, data_socialsig_aggregate, data_action_aggregate, data_sigurns, 
#              data_recurns, data_execurns, data_typed_time, data_signal_time, data_typed_time_norm, 
#              data_signal_time_norm, data_runid) = genBS_f7_RothErevExec_full_play_typed(mutation_rate, signal_cost, repmultipliers, 
#                                                                                         numprofiles, signal_snapshots, record_interval, genBSpunish, 
#                                                                                         numtypes, numsignals, runlength, numactions, 
#                                                                                         coordination_preferences, popt, sigurns, recurns, 
#                                                                                         population_size, sigdimensions, base_connection_weights, 
#                                                                                         sigmultipliers, homophily_factor, rgs[testruns], 
#                                                                                         epsilon, testruns)
#             final_simple_stat_002[testruns] = data_simple_stat_002
#             final_simple_stat_001[testruns] = data_simple_stat_001
#             final_socialsig_aggregate[testruns] = data_socialsig_aggregate
#             final_action_aggregate[testruns] = data_action_aggregate
#             final_sigurns[testruns] = data_sigurns
#             final_recurns[testruns] = data_recurns
#             final_execurns[testruns] = data_execurns
#             final_typed_time[testruns] = data_typed_time
#             final_signal_time[testruns] = data_signal_time
#             final_typed_time_norm[testruns] = data_typed_time_norm
#             final_signal_time_norm[testruns] = data_signal_time_norm
#             print(f'finished run #{testruns}')


        print('_____________________________________________________________________________')
        print(f'{runpass}_{dex0}_{dex1}') 
        print('payoffs =')
        print(coordination_preferences)
        print(f'punish = {genBSpunish}')
        print(f'signal cost = {signal_cost}')
        print(f'percent of agent types = {percent_agents_per_type}')
        print(f'homophily = {homophily_factor}')
        print(f'forgetting factor = {BZforget_multiplier}')
        final_simple_stat_001array = np.array(final_simple_stat_001)
        final_simple_stat_001mean = np.mean(final_simple_stat_001array, axis = 0)
        print(final_simple_stat_001mean)
        final_simple_stat_002array = np.array(final_simple_stat_002)
        final_simple_stat_002mean = np.mean(final_simple_stat_002array, axis = 0)
        print('final simple stat 2:')
        print(final_simple_stat_002mean)

        average_mutuinfo = average_mutual_info(numsignals, sigdimensions, sigmultipliers, numactions, final_sigurns, final_recurns, population_size, runs)
        print(f'average mutual information = {average_mutuinfo}')

        #         np.savez(f'{runpass}_{dex0}_{dex1}_.npz', final_mean=final_simple_stat_001mean, final_array2=final_simple_stat_002array, final_array=final_simple_stat_001array, final_socialsig=final_socialsig_aggregate, final_action=final_action_aggregate, final_sig=final_sigurns, final_rec=final_recurns, final_exec=final_execurns, final_typetime=final_typed_time, final_sigtime=final_signal_time, final_typetime_norm=final_typed_time_norm, final_sigtime_norm=final_signal_time_norm)

        type0 = 0
        type0opposition = np.array([1])
        type0agree = np.array([1])
        multidimsignalscount0, runsmultidimsignalscount0, oppositiondisruns0, agreeruns0, execsruns0, meanexecsruns0 = getalphasignaling(popt, final_execurns, type0, type0opposition, type0agree, sigmultipliers, population_size, final_sigurns, numsignals_perdim, sigdimensions, runs, numtypes, numsignals)


        print(' ')
        print('type 0 alphas:')
        print(multidimsignalscount0)

        print('execs attenton 0 alphas:')
        print(meanexecsruns0)
        print(' ')

        count_topA = 0
        count_topAact = 0
        count_topA2 = 0

        for concisedex in range(0, runs):
            cstat = final_simple_stat_001[concisedex]
            cstat2 = final_simple_stat_002[concisedex]
            for intdex7 in range(0, len(t0sigs_index)):
                if (np.argmax(final_socialsig_aggregate[concisedex][0][0]) == t0sigs_index[intdex7]) and (np.argmax(final_socialsig_aggregate[concisedex][0][1]) == t1sigs_index[intdex7]):
                    if np.argmax(final_socialsig_aggregate[concisedex][0][2]) == 0:
                        count_topA += 1
                        if cstat[0][0][0] == 1 and (cstat[1][0][0] == 1 and (cstat[1][1][1] == 1 and (cstat[0][2][2] == 1 and (cstat[1][2][2] == 1 and (cstat[2][2][2] == 1))))):
                            count_topAact += 1
                            if cstat2[0][0][0] == 1 and (cstat2[1][0][0] == 1 and (cstat2[1][1][1] == 1 and (cstat2[0][2][2] == 1 and (cstat2[1][2][2] == 1 and (cstat2[2][2][2] == 1))))):
                                count_topA2 += 1
        print(f'top A  count 0 = {count_topA}')
        concise0array[dex0][dex1] = count_topA
        print(f'top A  count 1 = {count_topAact}')
        concise1array[dex0][dex1] = count_topAact
        print(f'top A  count 2 = {count_topA2}')
        concise2array[dex0][dex1] = count_topA2


        finish = time.perf_counter()
        print(f'Finished in {round(finish-start,0)/60} minutes')
        print('^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^')
        print(' ')
        print(' ')
        # end indent
print('concise 0:')
print(concise0array)
print(' ')
print('concise 1:')
print(concise1array)
print(' ')
print('concise 2:')
print(concise2array)

# np.save(f'{runpass}_{dex0}_{dex1}_concise0_topA.npy', concise0array)
# np.save(f'{runpass}_{dex0}_{dex1}_concise1_topA.npy', concise1array)
# np.save(f'{runpass}_{dex0}_{dex1}_concise2_topA.npy', concise2array)


299283905714553442097129355798206586113
[[  0 151]
 [  1 179]
 [  2 170]]
finished run #10
finished run #0
finished run #20
finished run #30
finished run #40
finished run #60
finished run #50
finished run #70
finished run #80
finished run #90
_____________________________________________________________________________
genBS_v0055k05_BZforget3_assort_f7_RothErevExec_sweep_Merced_topA-Copy34_0_0
payoffs =
[[0.0075  0.      0.00425]
 [0.00625 0.0095  0.00425]
 [0.      0.      0.005  ]]
punish = [0. 0.]
signal cost = -0.000125
percent of agent types = [0.30000000000000004, 0.36, 0.34]
homophily = 0
forgetting factor = 0.998
[[[0.97 0.   0.03]
  [0.96 0.   0.04]
  [0.27 0.   0.73]]

 [[0.96 0.   0.04]
  [0.28 0.72 0.  ]
  [0.27 0.   0.73]]

 [[0.   0.   1.  ]
  [0.   0.   1.  ]
  [0.   0.   1.  ]]]
final simple stat 2:
[[[0.71 0.   0.03]
  [0.05 0.   0.  ]
  [0.27 0.   0.73]]

 [[0.69 0.   0.03]
  [0.05 0.53 0.  ]
  [0.26 0.   0.73]]

 [[0.   0.   0.27]
  [0.   0.   0.02]
  [0.   0.   1. 